# 03. Time Series Analysis

Bu notebook'ta Amazon hisse senedi kapanış fiyatı (Close) ve bu seriden üretilen çeşitli dönüşümler (First Difference, Percentage Return, Log Return) üzerinde zaman serisi analizi yapılmaktadır. Modelleme öncesi verinin durağanlığı (stationarity), otokorelasyon yapısı (ACF/PACF) ve temel baseline modellerin performansı incelenmektedir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Veri Yükleme ve Dönüşümler

Öncelikle veriyi yüklüyor ve 4 temel seriyi oluşturuyoruz:
- **Close Price:** Ham kapanış fiyatı.
- **First Difference:** $P_t - P_{t-1}$
- **Percentage Return:** $(P_t - P_{t-1}) / P_{t-1}$
- **Log Return:** $\ln(P_t / P_{t-1})$

In [ ]:
df = pd.read_csv('../data/amzn_stock.csv', index_col="Date", parse_dates=True)
close = df['Close']

# Serilerin hesaplanması
diff = close.diff().dropna()
pct = close.pct_change().dropna()
log_ret = np.log(close / close.shift(1)).dropna()

series_dict = {
    'Close Price': close,
    'First Difference': diff,
    'Percentage Return': pct,
    'Log Return': log_ret
}

print("Seriler başarıyla oluşturuldu.")

## 2. Descriptive Statistics & Visualizations

Her bir seri için temel istatistikleri hesaplıyor, zaman serisi grafiği, histogram, 30 günlük rolling mean ve rolling std hesaplamalarını görselleştiriyoruz.

In [ ]:
for name, s in series_dict.items():
    print(f"\n{'='*40}\n{name} - Descriptive Stats\n{'='*40}")
    print(s.describe())
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    
    # Time Series Plot + Rolling Stats
    axes[0].plot(s, label='Original', alpha=0.7)
    axes[0].plot(s.rolling(30).mean(), label='Rolling Mean (30d)', color='red')
    axes[0].set_title(f"{name} - Time Series")
    axes[0].legend()
    
    # Histogram
    axes[1].hist(s, bins=50, color='skyblue', edgecolor='black')
    axes[1].set_title(f"{name} - Histogram")
    
    plt.tight_layout()
    plt.show()
    
    # Rolling Std Plot
    plt.figure(figsize=(7, 3))
    plt.plot(s.rolling(30).std(), label='Rolling Std (30d)', color='orange')
    plt.title(f"{name} - Rolling Volatility")
    plt.legend()
    plt.show()

## 3. Stationarity Analizi

Zaman serisi modellerinin çoğu verinin durağan (stationary) olmasını varsayar. Durağanlığı test etmek için ADF (Augmented Dickey-Fuller) ve KPSS testlerini uyguluyoruz.

- **ADF Testi:** H0 (Seri birim köke sahiptir, durağan değildir). p-value < 0.05 ise H0 reddedilir (durağandır).
- **KPSS Testi:** H0 (Seri durağandır). p-value < 0.05 ise H0 reddedilir (durağan değildir).

In [ ]:
results = []

for name, s in series_dict.items():
    # ADF Test
    adf_stat, adf_p, _, _, _, _ = adfuller(s.dropna())
    
    # KPSS Test
    try:
        kpss_stat, kpss_p, _, _ = kpss(s.dropna(), regression='c', nlags='auto')
    except:
        kpss_p = np.nan
        
    stationary_interpretation = ""
    if adf_p < 0.05 and kpss_p >= 0.05:
        stationary_interpretation = "Stationary"
    elif adf_p >= 0.05 and kpss_p < 0.05:
        stationary_interpretation = "Non-Stationary"
    else:
        stationary_interpretation = "Mixed (Differencing may be needed)"
        
    results.append({
        'Series': name,
        'ADF p-value': adf_p,
        'KPSS p-value': kpss_p,
        'Stationary interpretation': stationary_interpretation
    })

stationarity_df = pd.DataFrame(results)
display(stationarity_df)

### Yorum
- **Close Price** serisinin birim kök barındırdığı (non-stationary) açıkça görülmektedir.
- Dönüşüm uygulanan **First Difference**, **Percentage Return** ve **Log Return** serilerinin tamamı durağandır (stationary). Derin öğrenme ve istatistiksel modeller için bu serilerin kullanımı çok daha uygundur.

## 4. Autocorrelation Analizi (ACF & PACF)

Geçmiş değerlerde bir sonraki değeri tahmin etmek için anlamlı bir *temporal dependency* (zamansal bağımlılık) olup olmadığını inceleyeceğiz.

In [ ]:
for name, s in series_dict.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(s.dropna(), lags=40, ax=axes[0], title=f"ACF - {name}")
    plot_pacf(s.dropna(), lags=40, ax=axes[1], title=f"PACF - {name}")
    plt.show()

### Yorum
- **Close Price:** ACF grafiği yavaşça azalan bir yapı gösteriyor, bu da non-stationary (rastgele yürüyüş benzeri) karakteristiğin klasik bir işaretidir. Geçmiş değerler arasındaki yüksek korelasyon aslında trendden kaynaklanmaktadır.
- **First Difference, Percentage & Log Return:** İlk birkaç gecikme (lag) dışında anlamlı bir otokorelasyon neredeyse yok (beyaz gürültüye/white noise'a çok yakın). Bu, geçmiş günlerin getirisine bakarak ertesi günün getirisini tahmin etmenin zor olduğunu gösteriyor (Etkin Piyasa Hipotezi).

## 5. Baseline Analizi

Deep learning modellerinin (`02_model_experiments.ipynb` içindeki RMSE ~10.09) gerçekten bir şey öğrenip öğrenmediğini anlamak için basit baseline modeller kuralım. Test dönemini önceki deneyle (veri setinin son %15'i) aynı tutacağız.

1. **Persistence (Naive Baseline):** `prediction[t] = actual[t-1]`
2. **Moving Average Baseline:** `prediction[t] = mean(actual[t-k : t-1])` (örneğin k=5, k=20)

In [ ]:
val_end = int(len(close) * 0.85)
test_actual = close.iloc[val_end:]

# Baseline 1: Persistence
# prediction[t] = actual[t-1]
persistence_pred = close.iloc[val_end-1 : -1].values

# Baseline 2: Moving Average 5-day
ma5 = close.rolling(5).mean()
ma5_pred = ma5.iloc[val_end-1 : -1].values

# Baseline 3: Moving Average 20-day
ma20 = close.rolling(20).mean()
ma20_pred = ma20.iloc[val_end-1 : -1].values

# Metrik hesaplamaları
baselines = {
    'Persistence': persistence_pred,
    'MA (5-day)': ma5_pred,
    'MA (20-day)': ma20_pred
}

rmse_results = []
for name, preds in baselines.items():
    rmse = np.sqrt(mean_squared_error(test_actual.values, preds))
    rmse_results.append({'Model': name, 'RMSE': rmse})
    
baseline_df = pd.DataFrame(rmse_results)
print("Baseline RMSE Karşılaştırması (Test Seti)")
display(baseline_df)

### Yorum
- Sadece `prediction[t] = actual[t-1]` (Naive Persistence) kullanıldığında RMSE çok düşük çıkmaktadır (~3.73). 
- Oysa önceki deneylerde kurulan LSTM/GRU modelleri (Close üzerinden) 5.35 - 10.00 civarında RMSE üretmektedir. Bu da derin öğrenme modelinin persistence (naive) kadar bile iyi öğrenemediğini (veya trendi kaçırdığını / aşırı gecikmeli tahmin yaptığını) göstermektedir.

## 6. Target Recommendation

Analizler ışığında değerlendirmemiz:

* **Hangileri stationary?**
  * `First Difference`, `Percentage Return`, ve `Log Return` serileri durağandır (stationary). `Close Price` ise non-stationary'dir.
* **Hangilerinde autocorrelation var?**
  * `Close Price` serisinde yüksek ve yavaş sönen bir otokorelasyon vardır (trende bağlı). 
  * Diğer 3 dönüşüm (fark/getiri) serilerinde otokorelasyon çok zayıftır; veri büyük ölçüde beyaz gürültü (white noise) gibi davranmaktadır.
* **Raw Close prediction neden problemli?**
  * Seri non-stationary olduğu için mean ve variance zamanla değişir. 
  * Model, örüntü öğrenmek yerine sadece son değeri (veya son değerlerin ortalamasını) tekrar etmeyi (persistence) öğrenmeye eğilimlidir. Bu nedenle grafikte tahminler genellikle gerçekleşen değerin 1 gün kaydırılmış hali (lagged) gibi görünür ve RMSE naive modelden bile kötü çıkabilir.
* **Deep learning için hangi target daha mantıklı görünüyor?**
  * İstatistiksel özelliklerinin stabil olması (stationary) sebebiyle, **Log Return** veya **Percentage Return** (ikisi pratikte çok yakındır) hedefleri sinir ağları için çok daha mantıklıdır. Modelin mutlak bir seviyeyi değil, "oransal değişimi" tahmin etmesi istenir.
* **Bir sonraki deneyde hangi target kullanılmalı?**
  * **Log Return** kullanılması önerilir. Log return'lerin toplanabilir (additive) olması, simetrik yapısı ve aykırı değerlere (outliers) bir miktar daha dirençli olması model eğitimini kolaylaştırır. Tahmin edilen log return'ler, test aşamasında son gerçek Close fiyatı ile (veya iteratif olarak) birleştirilerek kolayca gerçek fiyata (Close) dönüştürülebilir.